In [ ]:
from dotenv import load_dotenv
from databricks.sdk import WorkspaceClient
load_dotenv(override=True)
w = WorkspaceClient()
for e in w.serving_endpoints.list():
    print(e.name, "|", e.task) - 

In [ ]:
import os, re
from dotenv import load_dotenv
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

load_dotenv(override=True)
w = WorkspaceClient()

MODEL = "databricks-meta-llama-3-3-70b-instruct"
C, S = "workspace", "ai_data_team"
wh = next(iter(w.warehouses.list()))

def run(sql):
    r = w.statement_execution.execute_statement(
        statement=sql, warehouse_id=wh.id, wait_timeout="50s")
    if r.status.state.value != "SUCCEEDED":
        raise RuntimeError(r.status.error.message)
    return r

def llm(system, user, max_tokens=600):
    r = w.serving_endpoints.query(
        name=MODEL, max_tokens=max_tokens, temperature=0,
        messages=[ChatMessage(role=ChatMessageRole.SYSTEM, content=system),
                  ChatMessage(role=ChatMessageRole.USER, content=user)])
    return r.choices[0].message.content

# --- Tell the agent what data exists ---
cols = run(f"""SELECT table_name, column_name, data_type
               FROM {C}.information_schema.columns
               WHERE table_schema = '{S}' AND table_name LIKE 'clean_%'
               ORDER BY table_name, ordinal_position""").result.data_array
schema = {}
for t, c, d in cols:
    schema.setdefault(t, []).append(f"{c} ({d})")
SCHEMA_TEXT = "\n".join(f"{C}.{S}.{t}: " + ", ".join(cs) for t, cs in schema.items())

DATA_NOTES = """
- Amounts are in USD. Revenue = SUM(total_amount_usd) from clean_orders.
- Use clean_orders for revenue and order counts. Use order_date for anything over time.
- Do NOT use registration_date for tenure or signup cohorts: most orders predate it (data quirk).
- clean_monthly_revenue and clean_product_summary are pre-aggregated; prefer clean_orders for custom cuts.
"""

SQL_SYSTEM = f"""You write Databricks SQL for a business analyst agent.
Tables:
{SCHEMA_TEXT}
Notes:{DATA_NOTES}
Return ONLY one SELECT query, no explanation, no markdown."""

FORBIDDEN = re.compile(r"\b(insert|update|delete|drop|create|alter|merge|truncate|grant|revoke)\b", re.I)

def write_sql(question):
    sql = llm(SQL_SYSTEM, question)
    sql = re.sub(r"^```(?:sql)?|```$", "", sql.strip(), flags=re.M).strip().rstrip(";")
    if not re.match(r"^(select|with)\b", sql, re.I) or FORBIDDEN.search(sql) or ";" in sql:
        raise ValueError(f"Blocked unsafe or invalid SQL:\n{sql}")
    return sql

def ask(question, show_sql=True):
    sql = write_sql(question)
    r = run(f"SELECT * FROM ({sql}) LIMIT 200")
    names = [c.name for c in r.manifest.schema.columns]
    rows = r.result.data_array if r.result and r.result.data_array else []
    if show_sql:
        print("SQL used:\n", sql, "\n")
    print(llm(
        "You explain data results to a non-technical business client. Use plain English, "
        "lead with the answer, quote the key numbers (USD), keep it under 120 words, "
        "and never invent numbers that are not in the results. If results are empty, say so.",
        f"Question: {question}\nColumns: {names}\nRows: {rows[:50]}\nData notes: {DATA_NOTES}"))

ask("what are the products are best sellers in last 12 monrhs, mention only top 5 products only")